In [1]:
# Pipeline Configuration
RUN_MODE = "production"  # "test" or "production"
TEST_SAMPLE_SIZE = 10
MAX_WORKERS = 8
CHUNK_SIZE = 1000
MAX_RETRIES = 5
TIMEOUT = 30
ENABLE_CACHE = True
ENABLE_CHECKPOINT = True
SAVE_PREVIEW_IMAGES = True
PIPELINE_VERSION = "1.0.0"


In [2]:
# !pip -q install geopandas rasterio rioxarray pystac-client planetary-computer odc-stac shapely pyproj xarray folium leafmap

In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np

from shapely.geometry import Point

import rasterio
import rioxarray

import planetary_computer
import pystac_client

In [4]:
import os

folders = [
    "../data",
    "../data/raw",
    "../data/processed",
    "../data/features",
    "../data/metadata",
    "../data/final",
    "../data/lucas",
    "../data/sentinel",
    "../data/weather",
    "../data/soilgrids",
    "../outputs",
    "../outputs/csv",
    "../outputs/maps",
    "../outputs/figures",
    "../outputs/reports",
    "../outputs/metrics",
    "../outputs/learning_curves",
    "../outputs/feature_importance",
    "../outputs/confusion_matrix",
    "../models",
    "../models/machine_learning",
    "../models/deep_learning",
    "../models/ensemble",
    "../models/best",
    "../models/experimental"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")


Created: ../data
Created: ../data/raw
Created: ../data/processed
Created: ../data/features
Created: ../data/metadata
Created: ../data/final
Created: ../data/lucas
Created: ../data/sentinel
Created: ../data/weather
Created: ../data/soilgrids
Created: ../outputs
Created: ../outputs/csv
Created: ../outputs/maps
Created: ../outputs/figures
Created: ../outputs/reports
Created: ../outputs/metrics
Created: ../outputs/learning_curves
Created: ../outputs/feature_importance
Created: ../outputs/confusion_matrix
Created: ../models
Created: ../models/machine_learning
Created: ../models/deep_learning
Created: ../models/ensemble
Created: ../models/best
Created: ../models/experimental


In [5]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

print("Connected Successfully")

Connected Successfully


In [6]:
# # collections = catalog.get_collections()

# # for c in collections:
    # # print(c.id)
print("Skipping collections listing for speed.")




Skipping collections listing for speed.


In [7]:
from shapely.geometry import Point

lon = 31.083
lat = 30.563

point = Point(lon, lat)

buffer = point.buffer(0.001)

geometry = buffer.__geo_interface__

geometry

{'type': 'Polygon',
 'coordinates': (((31.084, 30.563),
   (31.08399518472667, 30.562901982859668),
   (31.083980785280403, 30.562804909677983),
   (31.08395694033573, 30.562709715322743),
   (31.08392387953251, 30.562617316567632),
   (31.083881921264346, 30.56252860326317),
   (31.0838314696123, 30.56244442976698),
   (31.08377301045336, 30.562365606715836),
   (31.083707106781183, 30.562292893218814),
   (31.08363439328416, 30.562226989546637),
   (31.083555570233017, 30.562168530387698),
   (31.083471396736826, 30.56211807873565),
   (31.083382683432365, 30.562076120467488),
   (31.083290284677254, 30.562043059664266),
   (31.083195090322015, 30.562019214719594),
   (31.08309801714033, 30.562004815273326),
   (31.083, 30.561999999999998),
   (31.082901982859667, 30.562004815273326),
   (31.082804909677982, 30.562019214719594),
   (31.082709715322743, 30.562043059664266),
   (31.08261731656763, 30.562076120467488),
   (31.08252860326317, 30.56211807873565),
   (31.08244442976698, 30

In [8]:
lucas = pd.read_csv(
    "../data/raw/lucas/LUCAS-SOIL-2018/LUCAS-SOIL-2018-v2/LUCAS-SOIL-2018.csv",
    low_memory=False
)

print("Shape:", lucas.shape)

lucas.head()

Shape: (18984, 27)


,Depth,POINTID,pH_CaCl2,pH_H2O,EC,OC,CaCO3,P,N,K,...,NUTS_3,TH_LAT,TH_LONG,SURVEY_DATE,Elev,LC,LU,LC0_Desc,LC1_Desc,LU1_Desc
0,0-20 cm,47862690,4.1,4.81,8.73,12.4,3,< LOD,1.1,101.9,...,AT113,47.150238,16.134212,06-07-18,291,C23,U120,Woodland,Other coniferous woodland,Forestry
1,0-20 cm,47882704,4.1,4.93,5.06,16.7,1,< LOD,1.3,51.2,...,AT113,47.274272,16.175359,06-07-18,373,C21,U120,Woodland,Spruce dominated coniferous woodland,Forestry
2,0-20 cm,47982688,4.1,4.85,12.53,47.5,1,12.3,3.1,114.8,...,AT113,47.123260,16.289693,02-06-18,246,C33,U120,Woodland,Other mixed woodland,Forestry
3,0-20 cm,48022702,5.5,5.80,21.10,28.1,3,< LOD,2,165.8,...,AT113,47.245693,16.357506,06-07-18,305,C22,U120,Woodland,Pine dominated coniferous woodland,Forestry
4,0-20 cm,48062708,6.1,6.48,10.89,19.4,2,< LOD,2.2,42.1,...,AT113,47.296372,16.416782,05-07-18,335,C22,U120,Woodland,Pine dominated coniferous woodland,Forestry


In [9]:
import pandas as pd
import os

# Auto detect columns
point_id_col = [c for c in lucas.columns if 'pointid' in c.lower() or 'point_id' in c.lower()][0]
lat_col = [c for c in lucas.columns if 'lat' in c.lower() or 'latitude' in c.lower() or 'th_lat' in c.lower()][0]
lon_col = [c for c in lucas.columns if 'lon' in c.lower() or 'longitude' in c.lower() or 'long' in c.lower() or 'th_long' in c.lower()][0]
date_col = [c for c in lucas.columns if 'date' in c.lower() or 'survey_date' in c.lower()][0]

soil_vars = ['pH_H2O', 'pH_CaCl2', 'EC', 'OC', 'CaCO3', 'P', 'N', 'K']
soil_vars = [v for v in soil_vars if v in lucas.columns]

output_cols = [point_id_col, lat_col, lon_col, date_col] + soil_vars
rename_dict = {
    point_id_col: 'POINT_ID',
    lat_col: 'Latitude',
    lon_col: 'Longitude',
    date_col: 'Survey_Date'
}

lucas_cleaned = lucas[output_cols].rename(columns=rename_dict).dropna(subset=['POINT_ID'])

if RUN_MODE == "test":
    print(f"TEST mode active. Subsampling to first {TEST_SAMPLE_SIZE} records.")
    lucas_cleaned = lucas_cleaned.head(TEST_SAMPLE_SIZE)

os.makedirs("../data/features", exist_ok=True)
lucas_cleaned.to_csv("../data/features/lucas_labels.csv", index=False)
print("Exported lucas_labels.csv. Shape:", lucas_cleaned.shape)


Exported lucas_labels.csv. Shape: (18984, 12)
